In [4]:
import pandas as pd
import sqlite3 as sq
import matplotlib as mpl
from matplotlib import rcParams
import matplotlib.pyplot as plt
import numpy as np
import geopandas as gpd
import fiona
import os
import io
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
from src.functions import read_census_table, read_wp23_table

Read in base year from the Woods and Poole 2023 Vintage Database. The dependencies for these have been run, and the data is written out in the file "Employment Full Treatment.ipynb, found in the repository Woods-and-Poole-2023.

In [12]:
empbase = read_wp23_table('EmploymentBaseYear')

In [13]:
thelist = ['Christian County, Kentucky', 'Trigg County, Kentucky', 'Stewart County, Tennessee', 'Montgomery County, Tennessee']
empbase_filtered = empbase.loc[empbase['NAME'].isin(thelist)].reset_index(drop = True)

In [14]:
empbase_filtered.head()

,NAME,Education,Food Services,Government,Industrial,Medical,Office,Other,Retail,Service,Total employment (number of jobs)
0,"Christian County, Kentucky",657.0,2691.0,35865.0,10131.0,4313.0,7925.0,3140.0,4047.0,2725.0,71494.428571
1,"Montgomery County, Tennessee",3172.0,9784.0,9125.0,14732.0,10780.0,20778.0,7515.0,12400.0,8648.0,96933.785714
2,"Stewart County, Tennessee",44.0,291.0,986.0,830.0,224.0,644.0,1110.0,470.0,402.0,4998.357143
3,"Trigg County, Kentucky",54.0,417.0,704.0,604.0,557.0,1030.0,910.0,591.0,379.0,5247.678571


Now we're going to distribute this based on the same LEHD distribution of place of work employment by block for total jobs, with the most accurate representation of 2023, coming from the 2023 LODES 8 documents for S000 (all demographics), JT00 (all jobs). Here's the filestructure: https://lehd.ces.census.gov/data/lodes/LODES8/LODESTechDoc8.4.pdf

In [15]:
kylodes = pd.read_csv(r"https://lehd.ces.census.gov/data/lodes/LODES8/ky/wac/ky_wac_S000_JT00_2023.csv.gz")

In [18]:
kylodes.head(2)

,w_geocode,C000,CA01,CA02,CA03,CE01,CE02,CE03,"11 (Agriculture, Forestry, Fishing and Hunting)","21 (Mining, Quarrying, and Oil and Gas Extraction)",...,CFA02,CFA03,CFA04,CFA05,CFS01,CFS02,CFS03,CFS04,CFS05,createdate
0,210019701001004,1,0,0,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,20251202
1,210019701001018,3,0,1,2,1,2,0,0,0,...,0,0,0,0,0,0,0,0,0,20251202


In [22]:
coldict = {'CNS01': '11 (Agriculture, Forestry, Fishing and Hunting)',
'CNS02': '21 (Mining, Quarrying, and Oil and Gas Extraction)', 
'CNS03': '22 (Utilities)', 
'CNS04': '23 (Construction)', 
'CNS05': '31-33 (Manufacturing)', 
'CNS06': '42 (Wholesale Trade)', 
'CNS07': '44-45 (Retail Trade)', 
'CNS08': '48-49 (Transportation and Warehousing)', 
'CNS09': '51 (Information)', 
'CNS10': '52 (Finance and Insurance)', 
'CNS11': '53 (Real Estate and Rental and Leasing)', 
'CNS12': '54 (Professional, Scientific, and Technical Services)', 
'CNS13': '55 (Management of Companies and Enterprises)', 
'CNS14': '56 (Administrative and Support and Waste Management and Remediation Services)', 
'CNS15': '61 (Educational Services)', 
'CNS16': '62 (Health Care and Social Assistance)', 
'CNS17': '71 (Arts, Entertainment, and Recreation)', 
'CNS18': '72 (Accommodation and Food Services)', 
'CNS19': '81 (Other Services [except Public Administration])', 
'CNS20': '92 (Public Administration)', 
'C000': 'Total employment (number of jobs)'}

kylodes = kylodes.rename(columns = coldict)

In [23]:
kylodes_transp = kylodes[['w_geocode', 'Total employment (number of jobs)', '11 (Agriculture, Forestry, Fishing and Hunting)',  '21 (Mining, Quarrying, and Oil and Gas Extraction)', 
'22 (Utilities)', '23 (Construction)', '31-33 (Manufacturing)', '42 (Wholesale Trade)', '44-45 (Retail Trade)', '48-49 (Transportation and Warehousing)', 
'51 (Information)', '52 (Finance and Insurance)', '53 (Real Estate and Rental and Leasing)', '54 (Professional, Scientific, and Technical Services)', 
'55 (Management of Companies and Enterprises)', '56 (Administrative and Support and Waste Management and Remediation Services)', 
'61 (Educational Services)', '62 (Health Care and Social Assistance)', '71 (Arts, Entertainment, and Recreation)', 
'72 (Accommodation and Food Services)', '81 (Other Services [except Public Administration])', '92 (Public Administration)']].set_index('w_geocode').transpose()

In [25]:
kylodes_filt = kylodes[['w_geocode', 'Total employment (number of jobs)', '11 (Agriculture, Forestry, Fishing and Hunting)',  '21 (Mining, Quarrying, and Oil and Gas Extraction)', 
'22 (Utilities)', '23 (Construction)', '31-33 (Manufacturing)', '42 (Wholesale Trade)', '44-45 (Retail Trade)', '48-49 (Transportation and Warehousing)', 
'51 (Information)', '52 (Finance and Insurance)', '53 (Real Estate and Rental and Leasing)', '54 (Professional, Scientific, and Technical Services)', 
'55 (Management of Companies and Enterprises)', '56 (Administrative and Support and Waste Management and Remediation Services)', 
'61 (Educational Services)', '62 (Health Care and Social Assistance)', '71 (Arts, Entertainment, and Recreation)', 
'72 (Accommodation and Food Services)', '81 (Other Services [except Public Administration])', '92 (Public Administration)']]

In [26]:
kylodes_filt.head()

,w_geocode,Total employment (number of jobs),"11 (Agriculture, Forestry, Fishing and Hunting)","21 (Mining, Quarrying, and Oil and Gas Extraction)",22 (Utilities),23 (Construction),31-33 (Manufacturing),42 (Wholesale Trade),44-45 (Retail Trade),48-49 (Transportation and Warehousing),...,53 (Real Estate and Rental and Leasing),"54 (Professional, Scientific, and Technical Services)",55 (Management of Companies and Enterprises),56 (Administrative and Support and Waste Management and Remediation Services),61 (Educational Services),62 (Health Care and Social Assistance),"71 (Arts, Entertainment, and Recreation)",72 (Accommodation and Food Services),81 (Other Services [except Public Administration]),92 (Public Administration)
0,210019701001004,1,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,210019701001018,3,0,0,0,0,0,0,3,0,...,0,0,0,0,0,0,0,0,0,0
2,210019701001021,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,210019701001041,2,0,0,0,0,0,0,2,0,...,0,0,0,0,0,0,0,0,0,0
4,210019701002012,44,0,0,0,42,0,0,0,0,...,0,0,0,2,0,0,0,0,0,0


In [27]:
industry_mapping = {
    "11 (Agriculture, Forestry, Fishing and Hunting)": "Industrial",
    "21 (Mining, Quarrying, and Oil and Gas Extraction)": "Industrial",
    "22 (Utilities)": "Industrial",
    "23 (Construction)": "Industrial",
    "31-33 (Manufacturing)": "Industrial",
    "42 (Wholesale Trade)": "Industrial",
    "44-45 (Retail Trade)": "Retail",
    "48-49 (Transportation and Warehousing)": "Industrial",
    "51 (Information)": "Office",
    "52 (Finance and Insurance)": "Office",
    "53 (Real Estate and Rental and Leasing)": "Office",
    "54 (Professional, Scientific, and Technical Services)": "Office",
    "55 (Management of Companies and Enterprises)": "Office",
    "56 (Administrative and Support and Waste Management and Remediation Services)": "Service",
    "61 (Educational Services)": "Education",
    "62 (Health Care and Social Assistance)": "Medical",
    "71 (Arts, Entertainment, and Recreation)": "Service",
    "72 (Accommodation and Food Services)": "Food Services",
    "81 (Other Services [except Public Administration])": "Service",
    "92 (Public Administration)": "Government"
}

In [28]:
df_grouped = (kylodes_filt.rename(columns=industry_mapping).T.groupby(level=0).sum().T)

In [29]:
df_grouped.head()

,Education,Food Services,Government,Industrial,Medical,Office,Retail,Service,Total employment (number of jobs),w_geocode
0,0,0,0,1,0,0,0,0,1,210019701001004
1,0,0,0,0,0,0,3,0,3,210019701001018
2,0,0,0,0,1,0,0,0,1,210019701001021
3,0,0,0,0,0,0,2,0,2,210019701001041
4,0,0,0,42,0,0,0,2,44,210019701002012


Okay now we have the block level data and the control totals - time to distribute them for our base year allocation.